- 실습 기본 환경 설정


In [ ]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

# 10-2 트랜스포머의 변형 1, 인코더만 사용하는 트랜스포머

본 노트북은 본문 10-2절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 노이즈가 섞인 날짜 문자열의 형식을 여섯 가지 중 하나로 분류하는 데이터 준비
- 입력 앞에 `[CLS]` 토큰을 추가하는 데이터셋
- `nn.TransformerEncoderLayer`와 `nn.TransformerEncoder`로 만드는 `DateFormatClassifier`
- 노이즈 길이 유형별 분류 정확도와 평균 신뢰도([표 10-7])
- `[CLS]` 토큰이 어디를 보고 있는지 셀프 어텐션으로 확인

본문의 모델 13의 구현에 해당되며, 모델 13의 제시문은 다음과 같다.

> **모델 13. 날짜 형식 분류기 모델**
>
> 날짜 문자열의 앞뒤에 노이즈가 섞인 입력을 받아, 그 안의 날짜가 여섯 가지 형식 중 어느 것인지 분류하는 모델을 만들어 본다.

## 데이터 생성

- 데이터를 만들기 전에 정답을 판정하는 형식 판별기부터 만든다.
    - 생성기가 정답을 주장하는 것이 아니라, 생성기가 만든 문자열을 판별기가 검사해 형식이 유일할 때만 데이터로 채택한다.
- 한 문자열이 두 형식에 동시에 해당하는 일이 없어야 과제가 성립한다.
    - 5월은 전체 이름과 약어가 모두 `May`라서, 약어 형식을 함께 쓰면 형식이 유일하게 결정되지 않는다. 그래서 여섯 형식은 월 이름을 전체 이름으로만 사용한다.

In [ ]:
# 참고 - 데이터 생성을 위한 함수

import re
import random
import string
import unicodedata
from calendar import monthrange
from collections import Counter, namedtuple

# 로케일에 영향받지 않도록 월 이름을 상수로 고정
MONTH_FULL = ['', 'January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
MONTH_ABBR = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 월 이름을 전체 이름으로만 쓰는 여섯 형식(레이블 0~5 순서 유지)
SRC_FORMATS = [
    '%d %B %Y',     # 01 February 2026
    '%B %d, %Y',    # February 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]

NUM_FORMATS = len(SRC_FORMATS)
YEAR_RANGE = (1900, 2050)
MAX_INPUT_LENGTH = 40       # 노이즈를 포함한 입력 문자열의 최대 길이

# (연, 월, 일)을 fmt 형식에 맞춰 날짜 문자열로 변환
def render(year, month, day, fmt):
    text = fmt.replace('%Y', f'{year:04d}').replace('%B', MONTH_FULL[month])
    return text.replace('%m', f'{month:02d}').replace('%d', f'{day:02d}')

# 형식 문자열을 정규 표현식으로 변환하는 함수(%m, %d는 반드시 두 자리)
def _format_to_pattern(fmt):
    parts = []
    for token in re.split(r'(%[YmdB])', fmt):
        if token == '%Y':
            parts.append(r'(?P<year>\d{4})')
        elif token == '%m':
            parts.append(r'(?P<month>\d{2})')
        elif token == '%d':
            parts.append(r'(?P<day>\d{2})')
        elif token == '%B':
            parts.append('(?P<month_full>' + '|'.join(MONTH_FULL[1:]) + ')')
        else:
            parts.append(re.escape(token))
    return re.compile(''.join(parts))


FORMAT_PATTERNS = [(fmt, _format_to_pattern(fmt)) for fmt in SRC_FORMATS]

# 실제 존재하는 날짜인지 확인하는 함수
def is_real_date(year, month, day):
    if not YEAR_RANGE[0] <= year <= YEAR_RANGE[1]:
        return False
    if not 1 <= month <= 12:
        return False
    return 1 <= day <= monthrange(year, month)[1]

# 입력 문자열에서 발견되는 날짜 문자열 형식을 검색해 유일 형식인지를 판단하는 함수
def formats_in(text):
    found = set()
    for index, (fmt, pattern) in enumerate(FORMAT_PATTERNS):
        for matched in pattern.finditer(text):
            group = matched.groupdict()
            month = (MONTH_FULL.index(group['month_full'])
                     if group.get('month_full') else int(group['month']))
            if is_real_date(int(group['year']), month, int(group['day'])):
                found.add(index)
                break
    return found

- 월과 일의 모든 조합을 빠짐없이 만들어 형식이 서로 구별되는지 전수 검사한다.

In [ ]:
# 참고 - 출력 폭을 맞추기 위한 도우미 함수

# 한글처럼 두 칸을 차지하는 글자를 감안해 출력 폭을 맞출 때 사용
def pad(text, width, align='<'):
    display_width = sum(
        2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in str(text)
    )
    space = ' ' * max(0, width - display_width)
    return space + str(text) if align == '>' else str(text) + space

In [ ]:
# 참고 - 생성한 데이터의 형식이 유일하게 구분되는지 확인
# 1. 노이즈가 없는 상태에서의 검사
from datetime import date, timedelta

# 2024년(윤년)과 2025년(평년)의 모든 날짜 = 월/일의 모든 조합
check_dates = []
for year in (2024, 2025):
    day = date(year, 1, 1)
    while day.year == year:
        check_dates.append(day)
        day += timedelta(days=1)

ambiguous = []
for day in check_dates:
    for index, fmt in enumerate(SRC_FORMATS):
        text = render(day.year, day.month, day.day, fmt)
        if formats_in(text) != {index}:
            ambiguous.append((text, fmt, formats_in(text)))

print(f'검사한 문자열 {len(check_dates) * NUM_FORMATS}개 '
      f'(날짜 {len(check_dates)}개 x 형식 {NUM_FORMATS}개)')
print(f'두 형식 이상에 해당하는 문자열: {len(ambiguous)}개')
assert not ambiguous, ambiguous[:5]
print('여섯 형식은 표면만으로 완전히 구별된다.\n')

sample_day = date(2026, 2, 1)
print(pad('레이블', 8) + pad('형식', 12) + '예시')
print('-' * 44)
for index, fmt in enumerate(SRC_FORMATS):
    print(pad(index, 8) + pad(fmt, 12)
          + repr(render(sample_day.year, sample_day.month, sample_day.day, fmt)))

- 9-3절의 방식으로 날짜 문자열 앞뒤에 무작위 노이즈를 덧붙여 최대 40자로 만든다.
    - 여기서 새로운 위험이 생긴다. 노이즈가 우연히 다른 형식의 날짜 문자열을 만들어 낼 수 있다.
    - 그래서 노이즈를 붙인 뒤에도 판별기로 다시 검사한다.

In [ ]:
# 참고 - 노이즈 추가 함수
NOISE_CHARS = (
    string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
)
def add_random_noise(text, rng, max_length=MAX_INPUT_LENGTH):
    """날짜 문자열 앞뒤에 무작위 길이의 무작위 노이즈를 덧붙인다.

    9-1, 9-3, 10-1절의 `add_random_noise()`와 같은 방식이며, 전역 난수
    생성기 대신 주입받은 생성기를 쓰는 점만 다르다.
    """
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = rng.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(rng.choices(NOISE_CHARS, k=prefix_length))
    if remaining > 0:
        suffix_length = rng.randint(0, remaining)
        suffix = ''.join(rng.choices(NOISE_CHARS, k=suffix_length))
    return prefix + text + suffix

In [ ]:
# 참고 - 생성한 데이터의 형식이 유일하게 구분되는지 확인
# 2. 무작위 노이즈를 추가한 상태에서의 검사

# 노이즈를 섞었을 때 레이블이 유일하지 않게 되는 비율 측정
probe_rng = random.Random(2026)
PROBE_SIZE = 20000
collision = 0
collision_examples = []
for _ in range(PROBE_SIZE):
    index = probe_rng.randrange(NUM_FORMATS)
    year = probe_rng.randint(*YEAR_RANGE)
    month = probe_rng.randint(1, 12)
    day = probe_rng.randint(1, monthrange(year, month)[1])
    noisy = add_random_noise(
        render(year, month, day, SRC_FORMATS[index]), probe_rng)
    found = formats_in(noisy)
    if found != {index}:
        collision += 1
        if len(collision_examples) < 3:
            collision_examples.append((noisy, SRC_FORMATS[index], found))

print(f'노이즈 입력 {PROBE_SIZE}개 중 형식이 유일하지 않은 것: '
      f'{collision}개 ({collision / PROBE_SIZE * 100:.3f}%)')
for noisy, fmt, found in collision_examples:
    print(f'  {noisy!r}  지정한 형식={fmt}  가능한 형식={[SRC_FORMATS[i] for i in sorted(found)]}')

- 여섯 형식에 균등하게 배분하고, 판별기로 형식이 유일한지 확인하며, 중복 입력은 버린다.
    - 본문과 같이 12,000개의 샘플을 64:16:20의 비율로 훈련, 검증, 평가 데이터셋으로 나눈다.

In [ ]:
# 참고 - 데이터셋 생성

# 데이터셋의 각 샘플을 나타내는 namedtuple
Sample = namedtuple('Sample', 'text label clean noise_length')

# 노이즈가 섞인 날짜 형식 분류 데이터셋을 생성
#   발생 가능한 형식 모호성을 피하기 위해 형식이 유일하게 결정되지 않는 경우 폐기
def build_dataset(total=12000, ratios=(0.64, 0.16, 0.20), seed=42):
    rng = random.Random(seed)
    seen = set()
    discarded = Counter()
    strata = []
    per_format = total // NUM_FORMATS
    for index, fmt in enumerate(SRC_FORMATS):
        samples = []
        while len(samples) < per_format:
            year = rng.randint(*YEAR_RANGE)
            month = rng.randint(1, 12)
            day = rng.randint(1, monthrange(year, month)[1])
            clean = render(year, month, day, fmt)
            text = add_random_noise(clean, rng)
            if text in seen:                    # 중복 제거
                discarded['중복'] += 1
                continue
            if formats_in(text) != {index}:     # 형식이 유일하지 않으면 폐기
                discarded['형식 모호'] += 1
                continue
            seen.add(text)
            samples.append(Sample(text, index, clean, len(text) - len(clean)))
        strata.append(samples)

    train, valid, test = [], [], []
    for group in strata:
        rng.shuffle(group)
        n_train = round(len(group) * ratios[0])
        n_valid = round(len(group) * ratios[1])
        train += group[:n_train]
        valid += group[n_train:n_train + n_valid]
        test += group[n_train + n_valid:]
    for split in (train, valid, test):
        rng.shuffle(split)
    return {'train': train, 'valid': valid, 'test': test}, discarded


dataset, discarded = build_dataset(total=12000)

print(pad('형식', 12) + ''.join(f'{name:>8s}' for name in dataset)
      + pad('평균 길이', 12, '>'))
print('-' * 48)
counters = {name: Counter(s.label for s in split) for name, split in dataset.items()}
all_samples = [s for split in dataset.values() for s in split]
for index, fmt in enumerate(SRC_FORMATS):
    lengths = [len(s.text) for s in all_samples if s.label == index]
    print(pad(fmt, 12) + ''.join(f'{counters[name][index]:>8d}' for name in dataset)
          + pad(f'{sum(lengths) / len(lengths):.1f}', 12, '>'))
print('-' * 48)
print(pad('합계', 12) + ''.join(f'{len(split):>8d}' for split in dataset.values()))

print(f'\n생성 중 폐기: {dict(discarded)}')
texts = {name: {s.text for s in split} for name, split in dataset.items()}
print(f"훈련-검증 중복 {len(texts['train'] & texts['valid'])}건, "
      f"훈련-테스트 중복 {len(texts['train'] & texts['test'])}건")
noise_lengths = [s.noise_length for s in all_samples]
print(f'노이즈 길이: 최소 {min(noise_lengths)}, 최대 {max(noise_lengths)}, '
      f'평균 {sum(noise_lengths) / len(noise_lengths):.1f}')
print('\n예시:')
for s in dataset['train'][:5]:
    print(f'  {s.text!r:<44s} -> {SRC_FORMATS[s.label]:<12s} (원본 {s.clean!r})')

- 형식별 평균 길이가 거의 같다는 점에 주목하자.
    - 날짜 문자열 자체의 길이는 형식마다 다르지만, 전체 길이를 40자로 맞추면서 노이즈 길이가 그 차이를 흡수한다.
    - 덕분에 모델이 전체 길이만 보고 형식을 맞히는 편법을 쓸 수 없다.

## 어휘 사전과 데이터셋

- 어휘 사전에 없는 토큰은 `[UNK]` 토큰으로 바꿔 '모르는 글자'로 취급한다.
    - 어휘 사전은 훈련 분할만으로 구축한다.
- 데이터 샘플은 입력과 정답(형식 레이블)으로 구성되며, 입력은 `[CLS]` 토큰으로 시작해야 한다.

In [ ]:
# 참고 - 어휘 사전 
# 	어휘 사전에 없는 토큰은 [UNK] 토큰으로 바꿔 '모르는 글자'로 취급

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

CLS_TOKEN, PAD_TOKEN, UNK_TOKEN = '[CLS]', '[PAD]', '[UNK]'
CLS_IDX, PAD_IDX, UNK_IDX = 0, 1, 2
special_tokens = {CLS_TOKEN: CLS_IDX, PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}


class Vocab:
    def __init__(self, texts, special):
        tokens = set()
        for text in texts:
            tokens.update(text)
        self.vocab = dict(special)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + len(special)
        self.itos = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        return [self.vocab.get(c, UNK_IDX) for c in text]

    def __len__(self):
        return len(self.vocab)

In [ ]:
######################################################################################
# 코드 10-9 - 입력 앞에 [CLS] 토큰을 추가한 데이터셋 정의
######################################################################################

class DateFormatDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.samples = []
        for text, label in zip(texts, labels):
            # 입력 앞에 [CLS] 토큰 추가
            ids = [CLS_IDX] + vocab.encode(text)
            self.samples.append((ids, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

- 입력 문자열 샘플을 패딩해 길이를 맞추고 텐서로 변환하는 배치 병합 함수를 사용해 데이터로더를 만든다.

In [ ]:
# 참고 - 배치 병합 함수
#   : 입력 문자열 샘플을 패딩해 길이를 맞추고 텐서로 변환하는 배치 병합 함수를 사용해 데이터로더를 생성

def collate_fn(batch):
    seq_batch, label_batch = zip(*batch)
    seq_tensors = [torch.LongTensor(s) for s in seq_batch]
    src_padded = pad_sequence(seq_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, torch.LongTensor(label_batch)

In [ ]:
# 참고 - 어휘 사전과 데이터로더 생성
# 어휘 사전은 훈련 분할만으로 구축한다
vocab = Vocab([s.text for s in dataset['train']], special_tokens)

BATCH_SIZE = 32
loaders = {}
for name, split in dataset.items():
    loaders[name] = DataLoader(
        DateFormatDataset([s.text for s in split], [s.label for s in split], vocab),
        batch_size=BATCH_SIZE, shuffle=(name == 'train'), collate_fn=collate_fn,
    )

letters = ''.join(sorted(t for t in vocab.vocab if len(t) == 1))
print(f'어휘 사전 크기: {len(vocab)} ([CLS] + [PAD] + [UNK] + 글자들)')
print(f'어휘 사전의 글자: {letters!r}')
unseen = ({c for s in dataset['valid'] + dataset['test'] for c in s.text}
          - set(vocab.vocab))
print(f'훈련셋에 없는 글자: {len(unseen)}개 {sorted(unseen)}')

## 학습 가능한 위치 인코딩

- 10-1절의 [코드 10-1]과 같다.

In [ ]:
# 참고 - 학습 가능한 위치 인코딩(10-1절 [코드 10-1]과 동일)
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, max_length, d_model):
        super().__init__()
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.activation = nn.Tanh()

    def forward(self, token_embedded):
        seq_length = token_embedded.size(1)
        positions = torch.arange(seq_length, device=token_embedded.device)
        pos_embedded = self.position_embedding(positions)
        return self.activation(token_embedded + pos_embedded)

## DateFormatClassifier 모델

- 토큰 임베딩 계층, 위치 인코딩 계층, 드롭아웃 계층은 입력 데이터만 사용한다는 점만 빼면 10-1절의 `DateConverterTransformer`와 같다.
- `nn.TransformerEncoderLayer`는 셀프 어텐션과 두 개의 선형 계층으로 구성된 트랜스포머 인코더의 기본 블록 하나를 정의한다.
    - 인자의 의미는 `nn.Transformer` 생성자의 인자와 같다.
- 이 기본 블록을 `nn.TransformerEncoder`에 전달하면 여러 층 쌓은 인코더 블록 계층이 만들어진다.
    - `num_layers` 인자는 `nn.Transformer`의 `num_encoder_layers`와 같은 의미다.
- `enable_nested_tensor=False`는 실험 단계 API인 중첩 텐서 사용을 비활성화해 경고 메시지를 막는다.
- 분류기는 모든 토큰의 출력 대신 `[CLS]` 토큰의 콘텍스트 벡터만 입력받아 여섯 형식 중 하나로 분류한다.
    - `[CLS]`의 출력으로만 형식을 판정하므로, 학습 과정에서 `[CLS]`의 출력이 자연스럽게 입력 전체의 맥락을 대표하도록 최적화된다.
- 디코더가 없는 구조이므로 입력 패딩 마스크만 사용한다.

In [ ]:
######################################################################################
# 코드 10-8 - 인코더만 사용하는 트랜스포머 모델, DateFormatClassifier 클래스
######################################################################################

class DateFormatClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_layers, max_length, num_classes, dropout):
        super().__init__()
        # 입력 토큰 임베딩, 위치 인코딩, 드롭아웃 (10-1 절과 동일)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

        # 트랜스포머(인코더)의 기본 블록
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True
        )
        # 기본 블록을 여러 개 쌓은 트랜스포머(인코더) 블록
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            # 중첩 텐서(nested tensor) API 사용 비활성화 - 실험적 API 경고 방지
            enable_nested_tensor=False
        )
        # 분류기 - [CLS] 토큰의 출력 벡터를 받아 여섯 형식 중 하나로 분류
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, source):
        # PAD 마스크 - Transformer 계열 클래스는 PAD 위치가 True 인 마스크 사용
        # UNK 는 마스크 대상이 아니므로 모르는 글자도 어텐션에 참여한다
        pad_mask = (source == PAD_IDX)
        x = self.dropout(self.pos_encoding(self.embedding(source)))
        output = self.encoder(x, src_key_padding_mask=pad_mask)
        # [CLS](0번 위치)의 출력 벡터를 사용해 형식 분류
        cls_output = output[:, 0, :]            # (B, d_model)
        return self.classifier(cls_output)

## 모델의 학습

- 정답이 순차 데이터가 아닌 분류 레이블이므로, 모델에는 입력만 전달하고 정답은 손실 계산에만 사용한다.
- 토큰 단위의 오차가 아니라 입력 전체의 형식으로 손실을 계산하므로, 손실 함수는 `ignore_index` 인자를 쓰지 않는다.

In [ ]:
# 참고 - 학습 함수

import copy

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, sample_size, correct_size = 0.0, 0, 0
    for src, labels in loader:
        src, labels = src.to(device), labels.to(device)
        optimizer.zero_grad()
        # 모델에는 입력 문자열만 전달(정답 레이블은 전달하지 않음)
        logits = model(src)                         # (B, NUM_FORMATS)
        # 정답은 손실을 계산하는 용도로만 사용, ignore_index 인자 사용하지 않음
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
        correct_size += (logits.argmax(1) == labels).sum().item()
    return loss_sum / sample_size, correct_size / sample_size * 100.0


@torch.no_grad()
def validation(model, loader, criterion, device):
    model.eval()
    loss_sum, sample_size, correct_size = 0.0, 0, 0
    for src, labels in loader:
        src, labels = src.to(device), labels.to(device)
        logits = model(src)
        loss = criterion(logits, labels)
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
        correct_size += (logits.argmax(1) == labels).sum().item()
    return loss_sum / sample_size, correct_size / sample_size * 100.0


def train_loop(model, train_loader, valid_loader, criterion, optimizer,
               epochs, patience, device):
    model.to(device)
    log = common.EpochLogger(epochs)
    best_valid_loss = float('inf')
    best_state, counter = None, 0
    stopped = False
    for epoch in range(1, epochs + 1):
        train_loss, _ = train_epoch(
            model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = validation(
            model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                stopped = True
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    log.summary(stopped='조기 종료' if stopped else None)
    return log

In [ ]:
# 참고 - 모델 객체 생성과 학습
import torch.optim as optim

# 모델 구조 하이퍼파라미터
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_LAYERS = 2
MAX_LENGTH = 48             # 입력 최대 40자 + [CLS] + 여유
DROPOUT = 0.1

# 학습 설정 하이퍼파라미터
LR = 1e-3
EPOCHS = 60
PATIENCE = 8

# 결과 재현을 위한 시드값 고정
common.set_seed(SEED)

# 모델 객체 생성
model = DateFormatClassifier(
    vocab_size=len(vocab), d_model=D_MODEL, num_heads=NUM_HEADS,
    ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=MAX_LENGTH,
    num_classes=NUM_FORMATS, dropout=DROPOUT
).to(device)

# 손실 함수와 옵티아미저 생성
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# 모델 학습 실행
log = train_loop(model, loaders['train'], loaders['valid'],
                 criterion, optimizer, EPOCHS, PATIENCE, device)

In [ ]:
# 참고 - 학습 곡선 시각화
log.plot(title='DateFormatClassifier 학습 곡선')

- 본문에서는 최적 46 에포크, 검증 손실 0.0673, 전체 학습 시간 1분 39초로 조기 종료했다.

## 평가

- 정답이 여섯 개로 닫혀 있으므로 혼동 행렬을 그릴 수 있다.

In [ ]:
# 참고 - 정확도 계산 및 혼동 행렬 시각화
accuracy, n_correct, n_samples = common.get_accuracy(model, loaders['test'], device)
print(f'테스트셋 정확도: {accuracy:.2f}% ({n_correct}/{n_samples})')

confusion_matrix = common.get_confusion_matrix(
    model, loaders['test'], num_classes=NUM_FORMATS, device=device)

print(f'\n{pad("형식", 12)}{pad("샘플 수", 10, ">")}{pad("정확도(%)", 12, ">")}')
print('-' * 34)
for index, fmt in enumerate(SRC_FORMATS):
    row = confusion_matrix[index]
    total = row.sum().item()
    print(pad(fmt, 12) + pad(total, 10, '>')
          + pad(f'{row[index].item() / total * 100:.1f}', 12, '>'))

viz.plot_confusion_matrix(confusion_matrix, class_names=SRC_FORMATS)

- 혼동 행렬은 어떤 형식을 어떤 형식으로 착각하는지 보여 주지만, 각 칸에 몇 건이 모여 있는지만 알려 준다.
    - 오분류 사례를 직접 꺼내 보면 무엇이 모델을 헷갈리게 했는지 알 수 있다.

In [ ]:
# 참고 - 오분류 사례 전체 출력(오류 유형별)

# 각 분할된 데이터셋 전체에 대한 예측 레이블과 신뢰도를 한 번에 계산
@torch.no_grad()
def predict_split(model, samples, vocab, device, batch_size=128):
    model.eval()
    preds, confidences = [], []
    for start in range(0, len(samples), batch_size):
        chunk = samples[start:start + batch_size]
        seq_tensors = [
            torch.LongTensor([CLS_IDX] + vocab.encode(s.text)) for s in chunk
        ]
        src = pad_sequence(
            seq_tensors, batch_first=True, padding_value=PAD_IDX).to(device)
        probs = torch.softmax(model(src), dim=1)
        confidence, pred = probs.max(dim=1)
        preds += pred.tolist()
        confidences += confidence.tolist()
    return preds, confidences

test_samples = dataset['test']
test_preds, test_confidences = predict_split(model, test_samples, vocab, device)

# 오분류를 (정답 형식, 예측 형식) 쌍으로 취합
errors = {}
for sample, pred, confidence in zip(test_samples, test_preds, test_confidences):
    if pred != sample.label:
        errors.setdefault((sample.label, pred), []).append((sample, confidence))

error_size = sum(len(cases) for cases in errors.values())
print(f'테스트셋 {len(test_samples)}개 중 오분류 {error_size}건, '
      f'(정답 -> 예측) 유형 {len(errors)}가지')

# 건수가 많은 유형부터, 유형 안에서는 신뢰도가 높은(더 확신하며 틀린) 순서로 출력
for (label, pred), cases in sorted(errors.items(), key=lambda kv: -len(kv[1])):
    print(f'[정답 {SRC_FORMATS[label]} -> 예측 {SRC_FORMATS[pred]}] {len(cases)}건')
    for sample, confidence in sorted(cases, key=lambda case: -case[1]):
        print(f'  {sample.text!r:<44s} 포함된 날짜 문자열 {sample.clean!r:<21s} '
              f'노이즈 {sample.noise_length:>2d}자  신뢰도 {confidence * 100:5.1f}%')
    print()

## 노이즈 길이별 정확도와 신뢰도([표 10-7])

- 이 과제의 난이도는 노이즈 길이가 결정한다. 평가 데이터셋을 노이즈 길이 구간으로 나눠 정확도와 평균 신뢰도를 함께 본다.

In [ ]:
# 참고 - 노이즈 길이 구간 별 정확도와 평균 신뢰도 계산

BUCKETS = [(0, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)]
print(pad('노이즈 길이', 14) + pad('샘플 수', 10, '>')
      + pad('정확도(%)', 12, '>') + pad('평균 신뢰도(%)', 16, '>'))
print('-' * 52)
for low, high in BUCKETS:
    picked = [
        (s, p, c) for s, p, c in zip(test_samples, test_preds, test_confidences)
        if low <= s.noise_length <= high
    ]
    if not picked:
        continue
    correct = sum(p == s.label for s, p, _ in picked)
    confidence_sum = sum(c for _, _, c in picked)
    label = f'{low}' if low == high else f'{low}~{high}'
    print(pad(label, 14) + pad(len(picked), 10, '>')
          + pad(f'{correct / len(picked) * 100:.1f}', 12, '>')
          + pad(f'{confidence_sum / len(picked) * 100:.1f}', 16, '>'))

## 신뢰도를 함께 출력하는 예측 함수

- 예측 함수도 여느 분류 모델과 같은 방식으로 작성하면 되는데, 이 과정에서 분류 결과와 함께 신뢰도도 구할 수 있다.
    - 로짓에 소프트맥스를 적용한 값 중 가장 큰 값이 신뢰도가 된다.

In [ ]:
######################################################################################
# 코드 10-10 - 분류 결과의 신뢰도를 계산하는 예측 함수
######################################################################################

@torch.no_grad()
def predict(model, input_text, vocab, device):
    model.eval()
    # 입력 앞에 [CLS] 토큰을 추가한 후, 배치 차원을 추가한 텐서를 예측에 사용
    ids = [CLS_IDX] + vocab.encode(input_text)
    src = torch.LongTensor(ids).unsqueeze(0).to(device)
    logits = model(src)
    # 소프트맥스 확률로 변환한 후, 예측 클래스와 그 확률(예측 신뢰도) 계산
    probs = torch.softmax(logits, dim=1)
    pred = logits.argmax(1).item()
    return pred, probs[0][pred].item()

# 같은 날짜를 여섯 형식으로 만들고, 같은 노이즈를 추가해 확인
demo_rng = random.Random(10)
test_day = date(1938, 10, 31)
print(pad('입력', 46) + pad('정답 형식', 13) + pad('예측 형식', 13)
      + pad('신뢰도', 7, '>') + '  정답 여부')
print('-' * 90)
for index, fmt in enumerate(SRC_FORMATS):
    clean = render(test_day.year, test_day.month, test_day.day, fmt)
    text = add_random_noise(clean, demo_rng)
    pred, confidence = predict(model, text, vocab, device)
    print(pad(repr(text), 46) + pad(fmt, 13) + pad(SRC_FORMATS[pred], 13)
          + pad(f'{confidence * 100:.1f}%', 7, '>') + ' ' * 10
          + ('O' if pred == index else 'X'))

## 참고 - [CLS]는 어디를 보고 있나

- `[CLS]`는 입력의 어느 글자도 아니므로, 분류에 필요한 정보를 셀프 어텐션으로 직접 끌어와야 한다.
- 트랜스포머 블록의 층별로 `[CLS]`의 셀프 어텐션 가중치를 뽑아 확인한다.

In [ ]:
# 참고 - [CLS]의 셀프 어텐션 추출(트랜스포머 블록 층별)

# 각 인코더 블록에서 [CLS] 토큰의 입력 토큰별 어텐션 수집
@torch.no_grad()
def get_cls_attention(model, input_text, vocab, device):
    model.eval()
    ids = [CLS_IDX] + vocab.encode(input_text)
    src = torch.LongTensor(ids).unsqueeze(0).to(device)
    pad_mask = (src == PAD_IDX)
    # 모델의 forward와 같은 순서로 입력을 생성(eval 모드이므로 드롭아웃은 항등 함수)
    x = model.dropout(model.pos_encoding(model.embedding(src)))

    rows = []
    for layer in model.encoder.layers:
        # 층이 실제로 수행하는 것과 같은 셀프 어텐션을 가중치와 함께 다시 계산
        _, weights = layer.self_attn(
            x, x, x, key_padding_mask=pad_mask,
            need_weights=True, average_attn_weights=True
        )
        rows.append(weights[0, 0])                   # [CLS] 행(헤드 평균)
        x = layer(x, src_key_padding_mask=pad_mask)  # 다음 층의 입력
    tokens = [CLS_TOKEN] + list(input_text)
    return torch.stack(rows).cpu(), tokens


# 노이즈가 충분히 긴, 맞게 분류한 테스트 샘플을 하나 선택
target = next(
    s for s, p in zip(test_samples, test_preds)
    if p == s.label and s.noise_length >= 20
)
attention, tokens = get_cls_attention(model, target.text, vocab, device)
pred, confidence = predict(model, target.text, vocab, device)

print(f'입력      : {target.text!r}')
print(f'날짜 구간 : {target.clean!r} (노이즈 {target.noise_length}자)')
print(f'예측      : {SRC_FORMATS[pred]} (신뢰도 {confidence * 100:.1f}%)')

# 날짜 문자열 구간에 배분된 어텐션의 비율을 층별로 계산
start = target.text.index(target.clean) + 1      # +1 은 맨 앞의 [CLS] 토큰
end = start + len(target.clean)
print(f'\n{pad("층", 8)}{pad("날짜 구간 어텐션 비중(%)", 26, ">")}')
print('-' * 34)
for layer_index, row in enumerate(attention, start=1):
    share = row[start:end].sum().item() / row.sum().item() * 100
    print(pad(f'{layer_index}층', 8) + pad(f'{share:.1f}', 26, '>'))
print(f'\n(날짜 구간은 전체 {len(tokens)}자 중 {len(target.clean)}자, '
      f'{len(target.clean) / len(tokens) * 100:.1f}%를 차지)')

viz.plot_attention(
    attention,
    source_tokens=tokens,
    target_tokens=[f'{i}층 [CLS]' for i in range(1, len(attention) + 1)],
    title='[CLS] 토큰의 층별 어텐션', figsize=(14, 3), font_scale=0.8,
)

## 정리

- 인코더만 사용하는 트랜스포머는 분류 과제에 적합하다. `[CLS]` 토큰의 출력 벡터가 입력 전체를 대표하도록 학습된다.
- `nn.TransformerEncoderLayer`로 기본 블록을 만들고, `nn.TransformerEncoder`로 여러 층 쌓는다.
- 데이터를 만들기 전에 정답 판별기를 만들어 전수 검사하면 설계 결함을 미리 잡을 수 있다.
- 노이즈가 길어질수록 정확도와 신뢰도가 함께 떨어진다. 신뢰도는 모델이 얼마나 확신하는지를 알려 주는 유용한 신호다.